In [1]:
import os 

In [2]:
os.chdir("../")

In [3]:
%pwd

'd:\\MLOPS\\Projects\\DataScienceProject1'

In [4]:
from dataclasses import dataclass
from src.DataScienceProject import logger
from src.DataScienceProject.pipeline.data_ingetion_pipeline import DataIngetionTrainingPipeline
from pathlib import Path

In [5]:
@dataclass
class ModelTrainerConfig:
  root_dir : Path
  train_data_path : Path
  test_data_path : Path
  model_name : str
  alpha : float
  l1_ratio : float
  target_column : str

In [6]:
from src.DataScienceProject.constants import *
from src.DataScienceProject.utils.common import read_yaml, create_directories

In [24]:
class ConfigurationManager:
  def __init__(
    self,
    config_filepath = CONFIG_FILE_PATH,
    params_filepath = PARAMS_FILE_PATH,
    schema_filepath = SCHEMA_FILE_PATH):

      self.config = read_yaml(config_filepath)
      self.params = read_yaml(params_filepath)
      self. schema = read_yaml(schema_filepath)

      create_directories([self.config.model_trainer.root_dir])
  
  def get_model_trainer_config(self) -> ModelTrainerConfig:
    config = self.config.model_trainer
    params = self.params.ElasticNet
    schema = self.schema.TARGET_COLUMN
    create_directories([self.config.model_trainer.root_dir])
    model_trainer_config = ModelTrainerConfig(
    root_dir=config.root_dir,
    train_data_path = config. train_data_path,
    test_data_path = config.test_data_path,
    model_name = config.model_name,
    alpha = params.alpha,
    l1_ratio = params.l1_ratio,
    target_column = schema.name
    )

    return model_trainer_config

In [22]:
import pandas as pd
import os
from src.DataScienceProject import logger
from sklearn.linear_model import ElasticNet
import joblib

In [21]:
class ModelTrainer:
  def __init__(self, config : ModelTrainerConfig):
    self.config = config

  def train_model(self):
    logger.info("Loading train and test data")
    train_data = pd.read_csv(self.config.train_data_path)
    test_data = pd.read_csv(self.config.test_data_path)

    logger.info("Splitting input and target feature from train and test data")
    train_x = train_data.drop(self.config.target_column, axis=1)
    train_y = train_data[self.config.target_column]

    test_x = test_data.drop(self.config.target_column, axis=1)
    test_y = test_data[self.config.target_column]

    logger.info("Initializing the ElasticNet model")
    model = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio)

    logger.info("Training the ElasticNet model")
    model.fit(train_x, train_y)

    logger.info("Saving the trained model")
    joblib.dump(model, os.path.join(self.config.root_dir, self.config.model_name))

In [26]:
try:
  config = ConfigurationManager()
  model_trainer_config = config.get_model_trainer_config()
  model_trainer_config = ModelTrainer(config=model_trainer_config)
  model_trainer_config.train_model()
except Exception as e:
  logger.exception(e)

[2026-05-20 06:00:35,734]: INFO: common :yaml file: config\config.yaml loaded successfully 
[2026-05-20 06:00:35,736]: INFO: common :yaml file: params.yaml loaded successfully 
[2026-05-20 06:00:35,738]: INFO: common :yaml file: schema.yaml loaded successfully 
[2026-05-20 06:00:35,739]: INFO: common :created directory at: artifacts/model_trainer 
[2026-05-20 06:00:35,739]: INFO: common :created directory at: artifacts/model_trainer 
[2026-05-20 06:00:35,740]: INFO: 2487661973 :Loading train and test data 
[2026-05-20 06:00:35,767]: INFO: 2487661973 :Splitting input and target feature from train and test data 
[2026-05-20 06:00:35,781]: INFO: 2487661973 :Initializing the ElasticNet model 
[2026-05-20 06:00:35,782]: INFO: 2487661973 :Training the ElasticNet model 
[2026-05-20 06:00:35,790]: INFO: 2487661973 :Saving the trained model 
